# 6) Data Preparation

Prepare features and labels for analytics or machine learning. Fit transformations only on training data, use pipelines where possible, and split time-dependent data chronologically when appropriate.

**Learning loop:** read the concept → run the example → inspect output → attempt the exercise → check the solution section at the end.


## Practice Dataset Setup

Run this setup cell first. It creates a reproducible sample dataset used by the techniques below.


In [ ]:
# 6.0 Practice Dataset Setup
# Definition: Create a reproducible sample dataset used across all examples in this notebook.
# Example: Build the sample DataFrame and confirm it loads before running section techniques.

# Import pandas
import pandas as pd
# Import required helpers from sklearn.model_selection
from sklearn.model_selection import train_test_split
# Import required helpers from sklearn.preprocessing
from sklearn.preprocessing import OneHotEncoder, StandardScaler
# Import required helpers from sklearn.compose
from sklearn.compose import ColumnTransformer
# Import required helpers from sklearn.pipeline
from sklearn.pipeline import Pipeline
# Import required helpers from sklearn.impute
from sklearn.impute import SimpleImputer
# Create the sample DataFrame
sales = pd.DataFrame({
    'Region': ['South', 'West', 'South', 'North', 'West', 'North', 'South', 'West'],
    'Quantity': [1, 2, 1, 3, 2, 1, 4, 2],
    'UnitPrice': [750, 25, 700, 45, 30, 40, 100, 25],
    'Discount': [0.0, 0.1, 0.0, 0.05, 0.0, 0.1, 0.05, 0.0]
})
# Run this transformation step
sales['Sales'] = sales['Quantity'] * sales['UnitPrice'] * (1 - sales['Discount'])
# Convert data type for consistent analysis
sales['HighSales'] = (sales['Sales'] >= 200).astype(int)
# Compute X
X = sales[['Region', 'Quantity', 'UnitPrice', 'Discount']]
# Compute y
y = sales['HighSales']


## 6.1 Data selection

**Concept:** Select the appropriate rows, features and target.



In [ ]:
# 6.1 Data Selection
# Definition: Perform data selection to understand and validate the dataset.
# Example: Run the cell and interpret the output against the section objective.

# Compute X
X = sales[['Region', 'Quantity', 'UnitPrice', 'Discount']]
# Compute y
y = sales['HighSales']
# Evaluate this expression output
print(X.head())
# Compute frequency distribution
print(y.value_counts())


## 6.2 Data type conversion

**Concept:** Ensure numeric, categorical, date and target types are correct.



In [ ]:
# 6.2 Data Type Conversion
# Definition: Verify inferred data types and identify fields requiring conversion.
# Example: Run the cell and interpret the output against the section objective.

# Run this transformation step
sales['Quantity'] = pd.to_numeric(sales['Quantity'], errors='coerce')
# Convert data type for consistent analysis
sales['Region'] = sales['Region'].astype('category')
# Evaluate this expression output
print(sales.dtypes)


## 6.3 Feature creation

**Concept:** Create meaningful predictors using business knowledge.



In [ ]:
# 6.3 Feature Creation
# Definition: Prepare robust model features and prevent data leakage.
# Example: Run the cell and interpret the output against the section objective.

# Run this transformation step
sales['OrderValueBeforeDiscount'] = sales['Quantity'] * sales['UnitPrice']
# Evaluate this expression output
print(sales[['Quantity', 'UnitPrice', 'OrderValueBeforeDiscount']].head())


## 6.4 Feature selection

**Concept:** Choose useful predictors without including the target or post-outcome leakage.



In [ ]:
# 6.4 Feature Selection
# Definition: Prepare robust model features and prevent data leakage.
# Example: Run the cell and interpret the output against the section objective.

# Compute feature_cols
feature_cols = ['Region', 'Quantity', 'UnitPrice', 'Discount']
# Compute X
X = sales[feature_cols]
# Evaluate this expression output
print(X.columns.tolist())


## 6.5 Categorical encoding

**Concept:** Encode categories for estimators that require numeric inputs.



In [ ]:
# 6.5 Categorical Encoding
# Definition: Prepare robust model features and prevent data leakage.
# Example: Run the cell and interpret the output against the section objective.

# Compute encoded
encoded = pd.get_dummies(sales['Region'], prefix='Region', dtype=int)
# Evaluate this expression output
print(encoded.head())


## 6.6 Numerical scaling

**Concept:** Scale numeric features when the algorithm benefits from comparable scales.



In [ ]:
# 6.6 Numerical Scaling
# Definition: Prepare robust model features and prevent data leakage.
# Example: Run the cell and interpret the output against the section objective.

# Compute scaler
scaler = StandardScaler()
# Fit transformer/model and transform data in one step
scaled = scaler.fit_transform(sales[['Quantity', 'UnitPrice', 'Discount']])
# Evaluate this expression output
print(pd.DataFrame(scaled, columns=['Quantity_scaled', 'UnitPrice_scaled', 'Discount_scaled']).round(2))


## 6.7 Outlier treatment

**Concept:** Use domain rules or robust statistical methods; do not automatically delete rare valid sales.



In [ ]:
# 6.7 Outlier Treatment
# Definition: Perform outlier treatment to understand and validate the dataset.
# Example: Run the cell and interpret the output against the section objective.

# Compute q1
q1 = sales['Sales'].quantile(0.25)
# Compute q3
q3 = sales['Sales'].quantile(0.75)
# Compute iqr
iqr = q3 - q1
# Compute outliers
outliers = sales.loc[(sales['Sales'] < q1 - 1.5*iqr) | (sales['Sales'] > q3 + 1.5*iqr)]
# Evaluate this expression output
print(outliers[['Sales']])


## 6.8 Schema alignment

**Concept:** Align columns, names and types before combining datasets.



In [ ]:
# 6.8 Schema Alignment
# Definition: Inspect column names, null counts, data types, and memory usage.
# Example: List column names and review df.info() output for structure checks.

# Compute expected
expected = ['Region', 'Quantity', 'UnitPrice', 'Discount']
# Compute aligned
aligned = X.reindex(columns=expected)
# Evaluate this expression output
print(aligned.columns.tolist())


## 6.9 Train / validation / test split

**Concept:** Separate model development data from final evaluation data.



In [ ]:
# 6.9 Train / Validation / Test Split
# Definition: Prepare robust model features and prevent data leakage.
# Example: Split records into training/validation/test subsets with a fixed random seed.

# Split data into training and testing subsets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.25, random_state=42, stratify=y)
# Evaluate this expression output
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)


## 6.10 Data leakage prevention

**Concept:** Fit imputers, encoders and scalers inside pipelines; split before fitting preprocessing.



In [ ]:
# 6.10 Data Leakage Prevention
# Definition: Prepare robust model features and prevent data leakage.
# Example: Run the cell and interpret the output against the section objective.

# Compute numeric
numeric = ['Quantity', 'UnitPrice', 'Discount']
# Compute categorical
categorical = ['Region']
# Compute preprocessor
preprocessor = ColumnTransformer([
 ('num', Pipeline([('imputer', SimpleImputer(strategy='median')), ('scale', StandardScaler())]), numeric),
 ('cat', Pipeline([('imputer', SimpleImputer(strategy='most_frequent')), ('encode', OneHotEncoder(handle_unknown='ignore'))]), categorical)
])
# Evaluate this expression output
print('Preprocessor created; fit it only on training data.')


## 6.11 Data versioning

**Concept:** Track the dataset snapshot, source, timestamp, schema and transformation code.



In [ ]:
# 6.11 Data Versioning
# Definition: Perform data versioning to understand and validate the dataset.
# Example: Run the cell and interpret the output against the section objective.

# Import required helpers from datetime
from datetime import datetime, timezone
# Compute metadata
metadata = {'dataset': 'sales', 'created_utc': datetime.now(timezone.utc).isoformat(), 'rows': len(sales), 'columns': sales.columns.tolist()}
# Evaluate this expression output
print(metadata)


## Recap
Completed core techniques for Data Preparation in process 6, with runnable examples and practice prompts.
